# Input and output

Every inference mode reads sites through one interface and writes annotations through one set of writers, so where the genotypes come from and which format the annotations take are independent choices. Here we build one inference, write it to three formats and read each back.


## Sources

The likelihood kernel consumes {class}`~ancestree.sites.Site` records, each holding the alleles at one position and the allele of every haplotype, a diploid sample contributing two. {class}`~ancestree.sources.CyVCF2Source`, {class}`~ancestree.sources.VcfZarrSource` and {class}`~ancestree.sources.TskitSource` yield them from a VCF or BCF, a VCF Zarr store and a tree sequence, and any of these, or a list of records, may be passed to an inference constructor. Below, the tree sequence and the VCF written from it yield identical records.


In [1]:
import ancestree as anc
import tskit

# The quickstart tree sequence, exported to a VCF under the same sample names.
ts = tskit.load("quickstart.trees")
ingroup = [f"i{i}" for i in range(6)]
outgroup = ["o0", "o1"]
with open("quickstart.vcf", "w") as vcf:
    ts.write_vcf(vcf, contig_id="1", individual_names=ingroup + outgroup)

# The same sites read from either source.
from_ts = list(anc.TskitSource(ts))
from_vcf = list(anc.CyVCF2Source("quickstart.vcf"))

# Position and tip alleles agree site by site.
print(from_vcf[0])
same = all(a.pos == b.pos and a.tip_alleles == b.tip_alleles
           for a, b in zip(from_ts, from_vcf))
print(f"{len(from_vcf)} sites from either source, identical: {same}")

INFO:ancestree.TskitSource: Reading variants from a tree sequence (224 sites, 8 samples)
INFO:ancestree.CyVCF2Source: Genotypes read as ploidy 1; pass ploidy= to override
INFO:ancestree.CyVCF2Source: Reading variants from quickstart.vcf (8 haplotype samples)


Site(chrom='1', pos=556, alleles=('G', 'C'), tip_alleles={'i0': 'G', 'i1': 'G', 'i2': 'G', 'i3': 'C', 'i4': 'C', 'i5': 'C', 'o0': 'G', 'o1': 'G'}, local_tree_handle=None, info={})
224 sites from either source, identical: True


In [2]:
assert same


## Writers

{meth}`Inference.to_vcf() <ancestree.inference.Inference.to_vcf>`, {meth}`Inference.to_zarr() <ancestree.inference.Inference.to_zarr>` and {meth}`Inference.to_arg() <ancestree.inference.Inference.to_arg>` take the stored posteriors and write the same annotations: a VCF with `AA`, `AA_prob` and `AA_post` `INFO` fields, a VCF Zarr store with the matching `variant_AA`, `variant_AA_prob` and `variant_AA_post` arrays at its root, and a tree sequence whose sites carry the MAP allele as their `ancestral_state`, with the mutations re-derived against it, and the posterior under an `ancestree` key of the site metadata. Run-level metadata the provenance does not carry passes through `info` on each writer, every key becoming an `AA_<key>` field or array.


An ARG-mode run over the quickstart tree sequence supplies the posteriors to write.


In [3]:
inf = anc.Inference.from_arg(
    ts, anc.JC69(), mu=5e-8,
    ingroup_samples=ingroup, outgroup_samples=outgroup,
)
res = list(inf.infer())


INFO:ancestree.ARGBasedInference: Inferring the ancestral allele at 224 sites over 29 local trees from the ARG
ARGBasedInference: 100%|██████████| 29/29 [00:00<00:00, 412.26 trees/s]
INFO:ancestree.ARGBasedInference: Focal node: reported at the ingroup_mrca; 4 tree(s) where the ingroup is not monophyletic, so its MRCA subtends outgroup tips


The stored posteriors are written to all three formats.


In [4]:
inf.to_vcf("annotated.vcf.gz", posteriors=res)
inf.to_zarr("annotated.vcz", posteriors=res)
inf.to_arg("annotated.trees", posteriors=res);


INFO:ancestree.VCFWriter: Wrote 224 annotated sites to annotated.vcf.gz
INFO:ancestree.ZarrWriter: Wrote 224 annotated sites to annotated.vcz
INFO:ancestree.TskitWriter: Wrote 224 annotated sites to annotated.trees


## Readers

{class}`~ancestree.readers.Reader` opens any of the three formats, and {meth}`Reader.head() <ancestree.readers.Reader.head>` lists the first annotated records as one table whichever format was written.


In [5]:
anc.Reader("annotated.vcf.gz").head(3)


chrom   pos  alleles  AA       A       C       G       T
1       556  G/C      G   0.0000  0.0005  0.9995  0.0000
1       927  G/T      T   0.0000  0.0000  0.0000  1.0000
1      1031  T/A      T   0.0005  0.0000  0.0000  0.9995

{meth}`Reader.grade() <ancestree.readers.Reader.grade>` scores the file against a truth as {meth}`Inference.grade() <ancestree.inference.Inference.grade>` does, reading the panel from the file's provenance. The node at which the true allele is read is its `focal` argument, the ingroup MRCA by default.


In [6]:
anc.Reader("annotated.vcf.gz").grade(ts)


224 sites: MAP recovery 99.1%; mean Brier = 0.016

In [7]:
assert anc.Reader("annotated.vcf.gz").grade(ts).map_recovery > 0.95


In [8]:
vcf_map = [a.aa for a in anc.Reader("annotated.vcf.gz").annotations()]
for path in ("annotated.vcz", "annotated.trees"):
    same = [a.aa for a in anc.Reader(path).annotations()] == vcf_map
    print(f"{path}: MAP alleles identical to the VCF: {same}")


annotated.vcz: MAP alleles identical to the VCF: True
annotated.trees: MAP alleles identical to the VCF: True


In [9]:
for path in ("annotated.vcz", "annotated.trees"):
    assert [a.aa for a in anc.Reader(path).annotations()] == vcf_map


## Provenance

Every writer records what produced the annotations, in the VCF header, the root attributes of the store and the provenance table of the tree sequence, and {meth}`Reader.provenance() <ancestree.readers.Reader.provenance>` reads that record back from any of the three formats, so a set of annotations can be traced to the run that made them.


In [10]:
anc.Reader("annotated.trees").provenance()


ancestree 0.1.1 (arg mode), 2026-09-15T08:52:06.746725+00:00
  model                             JC69
  prior                             StationaryPrior
  mu                                5e-08
  chrom                             1
  n_workers                         1
  focal                             ingroup_mrca
  n_ingroup                         6
  ingroup_samples                   ['i0', 'i1', 'i2', 'i3', 'i4', 'i5']
  outgroup_samples                  ['o0', 'o1']
  n_trees_ingroup_non_monophyletic  4
  n_segments_without_ingroup_mrca   0
  n_draws                           1

In [11]:
import shutil
from pathlib import Path

Path("annotated.vcf.gz").unlink(missing_ok=True)
Path("annotated.trees").unlink(missing_ok=True)
shutil.rmtree("annotated.vcz", ignore_errors=True)
Path("quickstart.vcf").unlink(missing_ok=True)
